# Feature Analysis for Stock Return Prediction

This notebook analyzes the effectiveness of the generated features for predicting the **percentage change (Return)** of Taiwan stocks.

We will use:
1.  **Correlation Heatmap**: To see which features are linearly correlated with the target.
2.  **Feature Importance (Random Forest)**: To see which features are most useful for a non-linear tree-based model.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

# Add src to path
sys.path.insert(0, os.path.abspath('src'))

from Feature_Engineering import prepare_training_data

## 1. Load Data with Return as Target

We set `target_col='Return'` to predict the percentage change.

In [ ]:
STOCK_ID = "2330" # TSMC

# Prepare data: Target is Next Day's Return
df = prepare_training_data(STOCK_ID, target_col='Return', target_shift=-1)

if df is None or df.empty:
    print("Error: No data found. Please check if the database exists and has data for this stock.")
else:
    print(f"Data Shape: {df.shape}")
    display(df[['Close', 'Return', 'Target']].head())

## 2. Correlation Analysis (Heatmap)

We calculate the correlation of all features with the `Target` (Next Day Return). 
Since there are many features (80+), we will plot the **Top 20 features** with the highest absolute correlation.

In [ ]:
if df is not None and not df.empty:
    # Calculate correlation matrix
    corr_matrix = df.corr()

    # Get correlation with Target
    target_corr = corr_matrix['Target'].drop('Target') # Drop target itself

    # Sort by absolute correlation
    top_features = target_corr.abs().sort_values(ascending=False).head(20)
    top_features_names = top_features.index.tolist()

    print("Top 20 Features by Correlation with Target:")
    print(target_corr[top_features_names])

    # Plot Heatmap of these top features
    plt.figure(figsize=(12, 10))
    top_corr_matrix = df[top_features_names + ['Target']].corr()
    sns.heatmap(top_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
    plt.title("Correlation Heatmap (Top 20 Features vs Target)")
    plt.show()

### Analysis of Correlation
- **High Correlation**: Indicates a strong linear relationship. If `Return_Lag_1` is high, it means momentum exists.
- **Low Correlation**: Doesn't mean the feature is useless! It might have a non-linear relationship captured by complex models.

## 3. Feature Importance (Random Forest)

Linear correlation misses non-linear patterns. A Random Forest model can tell us which features are actually used to split the data and reduce variance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

if df is not None and not df.empty:
    # Prepare X and y
    X = df.drop(columns=['Target'])
    y = df['Target']

    # Split data (Time Series split: Train on past, Test on future)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Train Random Forest
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    # Get Feature Importances
    importances = rf.feature_importances_
    feature_names = X.columns

    # Create DataFrame for plotting
    fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    fi_df = fi_df.sort_values(by='Importance', ascending=False).head(20)

    # Plot
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis')
    plt.title("Top 20 Features by Random Forest Importance")
    plt.show()

## 4. Conclusion

By comparing the **Correlation Heatmap** and **Random Forest Importance**, we can identify the most robust features.

- Features that appear in **both** lists are likely very strong predictors.
- Features only in **Random Forest** list likely have complex, non-linear relationships with the price change.
- **Lag Features** (e.g., `Return_Lag_1`, `Volume_Lag_1`) often show high importance in time-series prediction.